**ANÁLISIS EXPLORATORIO DE DATOS EDA**

El objetivo del análisis es comprender la estructura y comportamiento de los datos de clientes antes de desarrollar el modelo predictivo de conversión.

In [ ]:
# Importamos las librerías necesarias

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración visual de los gráficos
sns.set_theme(style="whitegrid")

print("Librerías cargadas correctamente")

In [ ]:
# Carga de datos local

df = pd.read_csv("../../../data/raw/train.csv")

print("Datos cargados:", df.shape)

In [ ]:
df.head()

In [ ]:
print("Cantidad de filas:", df.shape[0])
print("Cantidad de columnas:", df.shape[1])

In [ ]:
df.columns

In [ ]:
for i, columna in enumerate(df.columns, start=1):
    print(i, "-", columna)

In [ ]:
df.info()

In [ ]:
# Variables numéricas

variables_numericas = [
    "edad",
    "ingresos",
    "ratio_deuda_ingresos",
    "antiguedad_cuenta_meses",
    "numero_productos",
    "saldo_promedio",
    "dias_ultima_transaccion",
    "antiguedad_direccion_meses",
    "visitas_web_ultimos_90_dias",
    "distancia_sucursal_km",
    "dia_preferido_pago",
    "dias_ultima_interaccion"
]

# Variables categóricas

variables_categoricas = [
    "ocupacion",
    "region",
    "canal_adquisicion",
    "banda_riesgo",
    "dispositivo_principal"
]

# Variables booleanas

variables_booleanas = [
    "tiene_tarjeta_credito",
    "activo_movil",
    "es_nuevo_cliente",
    "tiene_prestamo",
    "tiene_seguro"
]

print("Variables numéricas:", len(variables_numericas))
print("Variables categóricas:", len(variables_categoricas))
print("Variables booleanas:", len(variables_booleanas))

In [ ]:
faltantes = df.isnull().sum()

tabla_faltantes = pd.DataFrame({
    "Variable": faltantes.index,
    "Valores faltantes": faltantes.values
})

tabla_faltantes

In [ ]:
print("Total de valores faltantes:", df.isnull().sum().sum())

In [ ]:
duplicados = df.duplicated().sum()

print("Cantidad de filas duplicadas:", duplicados)

In [ ]:
duplicados_cliente_mes = df.duplicated(
    subset=["id_cliente", "mes"]
).sum()

print("Clientes repetidos dentro del mismo mes:", duplicados_cliente_mes)

In [ ]:
clientes_unicos = df["id_cliente"].nunique()

print("Cantidad de clientes únicos:", clientes_unicos)

In [ ]:
print("Cantidad de registros:", len(df))
print("Cantidad de clientes únicos:", df["id_cliente"].nunique())

cuantos meses aparece cada cliente


In [ ]:
meses_por_cliente = (
    df.groupby("id_cliente")["mes"]
      .nunique()
)

meses_por_cliente.describe()

In [ ]:
# Contamos cuántos clientes aparecen en 1 mes, 2 meses, 3 meses, etc.
conteo_meses = meses_por_cliente.value_counts().sort_index()

plt.figure(figsize=(10,5))

ax = sns.barplot(
    x=conteo_meses.index,
    y=conteo_meses.values
)

plt.title("Cantidad de meses registrados por cliente")
plt.xlabel("Número de meses en los que aparece el cliente")
plt.ylabel("Cantidad de clientes")

# Mostrar el número encima de cada barra
for i, valor in enumerate(conteo_meses.values):
    ax.text(
        i,
        valor + 50,
        str(valor),
        ha="center",
        fontsize=9
    )

plt.tight_layout()
plt.show()

In [ ]:
resumen_numericas = df[variables_numericas].describe().T

resumen_numericas

In [ ]:
df["objetivo"].value_counts()

In [ ]:
df["objetivo"].value_counts(normalize=True) * 100

In [ ]:
resumen_objetivo = pd.DataFrame({
    "Cantidad": df["objetivo"].value_counts(),
    "Porcentaje": df["objetivo"].value_counts(normalize=True) * 100
})

resumen_objetivo

In [ ]:
plt.figure(figsize=(7,5))

ax = sns.countplot(
    data=df,
    x="objetivo"
)

plt.title("Distribución de la variable objetivo")
plt.xlabel("Conversión")
plt.ylabel("Cantidad de observaciones")

plt.xticks(
    [0,1],
    ["No convirtió", "Convirtió"]
)

for container in ax.containers:
    ax.bar_label(container)

plt.tight_layout()
plt.show()

ANALIZAR LA EDAD
Usamos todas las filas del dataset. Como un mismo cliente puede aparecer en varios meses, ese cliente se cuenta varias

In [ ]:
# Crear rangos de edad
df["rango_edad"] = pd.cut(
    df["edad"],
    bins=[20, 30, 40, 50, 60, 70, 80],
    labels=[
        "21-30",
        "31-40",
        "41-50",
        "51-60",
        "61-70",
        "71-80"
    ],
    include_lowest=True
)

# Contar clientes/observaciones por rango
conteo_edades = (
    df["rango_edad"]
    .value_counts()
    .sort_index()
)

conteo_edades

In [ ]:
plt.figure(figsize=(9,5))

ax = sns.barplot(
    x=conteo_edades.index,
    y=conteo_edades.values
)

plt.title("Distribución de clientes por rango de edad")
plt.xlabel("Rango de edad")
plt.ylabel("Cantidad de observaciones")

# Valores encima de las barras
for i, valor in enumerate(conteo_edades.values):
    ax.text(
        i,
        valor + 100,
        f"{valor:,}",
        ha="center",
        fontsize=9
    )

plt.tight_layout()
plt.show()

ANALISIS DE EDAD
primero dejamos una sola fila por cliente

In [ ]:
clientes_unicos_edad = (
    df.sort_values("mes")
      .drop_duplicates(
          subset="id_cliente",
          keep="last"
      )
)

In [ ]:
clientes_unicos_edad["rango_edad"] = pd.cut(
    clientes_unicos_edad["edad"],
    bins=[20, 30, 40, 50, 60, 70, 80],
    labels=[
        "21-30",
        "31-40",
        "41-50",
        "51-60",
        "61-70",
        "71-80"
    ],
    include_lowest=True
)

conteo_edades_clientes = (
    clientes_unicos_edad["rango_edad"]
    .value_counts()
    .sort_index()
)

In [ ]:
plt.figure(figsize=(9,5))

ax = sns.barplot(
    x=conteo_edades_clientes.index,
    y=conteo_edades_clientes.values
)

plt.title("Distribución de clientes únicos por rango de edad")
plt.xlabel("Rango de edad")
plt.ylabel("Cantidad de clientes")

for i, valor in enumerate(conteo_edades_clientes.values):
    ax.text(
        i,
        valor + 30,
        f"{valor:,}",
        ha="center",
        fontsize=9
    )

plt.tight_layout()
plt.show()

**Ingresos**

In [ ]:
# ============================================
# INGRESOS
# ============================================

# Crear rangos automáticos pero con etiquetas legibles
grupos_ingresos, limites_ingresos = pd.cut(
    df["ingresos"],
    bins=5,
    retbins=True
)

etiquetas_ingresos = []

for i in range(len(limites_ingresos) - 1):
    etiquetas_ingresos.append(
        f"{limites_ingresos[i]:,.0f} - {limites_ingresos[i+1]:,.0f}"
    )

df["rango_ingresos"] = pd.cut(
    df["ingresos"],
    bins=limites_ingresos,
    labels=etiquetas_ingresos,
    include_lowest=True
)

conteo_ingresos = (
    df["rango_ingresos"]
    .value_counts()
    .sort_index()
)

# GRÁFICO 1: Distribución
plt.figure(figsize=(11,5))

ax = sns.barplot(
    x=conteo_ingresos.index,
    y=conteo_ingresos.values
)

plt.title("Distribución de los ingresos")
plt.xlabel("Rango de ingresos")
plt.ylabel("Cantidad de observaciones")

plt.xticks(rotation=25)

for i, valor in enumerate(conteo_ingresos.values):
    ax.text(
        i,
        valor + 100,
        f"{valor:,}",
        ha="center"
    )

plt.tight_layout()
plt.show()


# GRÁFICO 2: Boxplot
plt.figure(figsize=(10,3))

sns.boxplot(
    data=df,
    x="ingresos"
)

plt.title("Detección de valores atípicos - Ingresos")
plt.xlabel("Ingresos")

plt.tight_layout()
plt.show()

**RADIO DEUDA(q tan grande es su deuda de acuerdo a lo q gana):0.20 ->significa que la deuda representa aproximadamente el 20 % de sus ingresos.**

In [ ]:
# ============================================
# RATIO DEUDA - INGRESOS
# ============================================

bins_ratio = [0, 0.2, 0.4, 0.6, 0.8, 1.0]

labels_ratio = [
    "0.00 - 0.20",
    "0.20 - 0.40",
    "0.40 - 0.60",
    "0.60 - 0.80",
    "0.80 - 1.00"
]

df["rango_ratio_deuda"] = pd.cut(
    df["ratio_deuda_ingresos"],
    bins=bins_ratio,
    labels=labels_ratio,
    include_lowest=True
)

conteo_ratio = (
    df["rango_ratio_deuda"]
    .value_counts()
    .sort_index()
)

# GRÁFICO 1
plt.figure(figsize=(10,5))

ax = sns.barplot(
    x=conteo_ratio.index,
    y=conteo_ratio.values
)

plt.title("Distribución del ratio deuda-ingresos")
plt.xlabel("Ratio deuda / ingresos")
plt.ylabel("Cantidad de observaciones")

for i, valor in enumerate(conteo_ratio.values):
    ax.text(
        i,
        valor + 100,
        f"{valor:,}",
        ha="center"
    )

plt.tight_layout()
plt.show()


# GRÁFICO 2
plt.figure(figsize=(10,3))

sns.boxplot(
    data=df,
    x="ratio_deuda_ingresos"
)

plt.title("Detección de valores atípicos - Ratio deuda-ingresos")
plt.xlabel("Ratio deuda / ingresos")

plt.tight_layout()
plt.show()

 **Antiguedad de cuenta**

In [ ]:
# ============================================
# ANTIGÜEDAD DE LA CUENTA
# ============================================

bins_antiguedad = [
    0,
    12,
    24,
    48,
    72,
    120,
    df["antiguedad_cuenta_meses"].max() + 1
]

labels_antiguedad = [
    "0 - 12 meses",
    "13 - 24 meses",
    "25 - 48 meses",
    "49 - 72 meses",
    "73 - 120 meses",
    "Más de 120 meses"
]

df["rango_antiguedad_cuenta"] = pd.cut(
    df["antiguedad_cuenta_meses"],
    bins=bins_antiguedad,
    labels=labels_antiguedad,
    include_lowest=True
)

conteo_antiguedad = (
    df["rango_antiguedad_cuenta"]
    .value_counts()
    .sort_index()
)

# GRÁFICO 1
plt.figure(figsize=(11,5))

ax = sns.barplot(
    x=conteo_antiguedad.index,
    y=conteo_antiguedad.values
)

plt.title("Distribución de la antigüedad de la cuenta")
plt.xlabel("Antigüedad de la cuenta")
plt.ylabel("Cantidad de observaciones")

plt.xticks(rotation=20)

for i, valor in enumerate(conteo_antiguedad.values):
    ax.text(
        i,
        valor + 100,
        f"{valor:,}",
        ha="center"
    )

plt.tight_layout()
plt.show()


# GRÁFICO 2
plt.figure(figsize=(10,3))

sns.boxplot(
    data=df,
    x="antiguedad_cuenta_meses"
)

plt.title("Detección de valores atípicos - Antigüedad de la cuenta")
plt.xlabel("Antigüedad de la cuenta (meses)")

plt.tight_layout()
plt.show()

**Numero de productos**

In [ ]:
# ============================================
# NÚMERO DE PRODUCTOS
# ============================================

conteo_productos = (
    df["numero_productos"]
    .value_counts()
    .sort_index()
)

# GRÁFICO 1
plt.figure(figsize=(8,5))

ax = sns.barplot(
    x=conteo_productos.index,
    y=conteo_productos.values
)

plt.title("Distribución según número de productos")
plt.xlabel("Número de productos")
plt.ylabel("Cantidad de observaciones")

for i, valor in enumerate(conteo_productos.values):
    ax.text(
        i,
        valor + 100,
        f"{valor:,}",
        ha="center"
    )

plt.tight_layout()
plt.show()


# GRÁFICO 2
plt.figure(figsize=(10,3))

sns.boxplot(
    data=df,
    x="numero_productos"
)

plt.title("Distribución del número de productos")
plt.xlabel("Número de productos")

plt.tight_layout()
plt.show()

**Saldo promedio**

In [ ]:
# ============================================
# SALDO PROMEDIO
# ============================================

grupos_saldo, limites_saldo = pd.cut(
    df["saldo_promedio"],
    bins=5,
    retbins=True
)

etiquetas_saldo = []

for i in range(len(limites_saldo) - 1):
    etiquetas_saldo.append(
        f"{limites_saldo[i]:,.0f} - {limites_saldo[i+1]:,.0f}"
    )

df["rango_saldo"] = pd.cut(
    df["saldo_promedio"],
    bins=limites_saldo,
    labels=etiquetas_saldo,
    include_lowest=True
)

conteo_saldo = (
    df["rango_saldo"]
    .value_counts()
    .sort_index()
)

# GRÁFICO 1
plt.figure(figsize=(11,5))

ax = sns.barplot(
    x=conteo_saldo.index,
    y=conteo_saldo.values
)

plt.title("Distribución del saldo promedio")
plt.xlabel("Rango de saldo promedio")
plt.ylabel("Cantidad de observaciones")

plt.xticks(rotation=25)

for i, valor in enumerate(conteo_saldo.values):
    ax.text(
        i,
        valor + 100,
        f"{valor:,}",
        ha="center"
    )

plt.tight_layout()
plt.show()


# GRÁFICO 2
plt.figure(figsize=(10,3))

sns.boxplot(
    data=df,
    x="saldo_promedio"
)

plt.title("Detección de valores atípicos - Saldo promedio")
plt.xlabel("Saldo promedio")

plt.tight_layout()
plt.show()

**Dias de la ultima transaccion**

In [ ]:
# ============================================
# DÍAS DESDE LA ÚLTIMA TRANSACCIÓN
# ============================================

bins_transaccion = [0, 30, 60, 90, 180, 365]

labels_transaccion = [
    "0 - 30 días",
    "31 - 60 días",
    "61 - 90 días",
    "91 - 180 días",
    "181 - 365 días"
]

df["rango_ultima_transaccion"] = pd.cut(
    df["dias_ultima_transaccion"],
    bins=bins_transaccion,
    labels=labels_transaccion,
    include_lowest=True
)

conteo_transaccion = (
    df["rango_ultima_transaccion"]
    .value_counts()
    .sort_index()
)

# GRÁFICO 1
plt.figure(figsize=(10,5))

ax = sns.barplot(
    x=conteo_transaccion.index,
    y=conteo_transaccion.values
)

plt.title("Distribución de días desde la última transacción")
plt.xlabel("Días desde la última transacción")
plt.ylabel("Cantidad de observaciones")

for i, valor in enumerate(conteo_transaccion.values):
    ax.text(
        i,
        valor + 100,
        f"{valor:,}",
        ha="center"
    )

plt.tight_layout()
plt.show()


# GRÁFICO 2
plt.figure(figsize=(10,3))

sns.boxplot(
    data=df,
    x="dias_ultima_transaccion"
)

plt.title("Detección de valores atípicos - Última transacción")
plt.xlabel("Días desde la última transacción")

plt.tight_layout()
plt.show()

**Antiguedad de direccion**

In [ ]:
# ============================================
# ANTIGÜEDAD DE LA DIRECCIÓN
# ============================================

bins_direccion = [
    0,
    12,
    24,
    48,
    72,
    120,
    df["antiguedad_direccion_meses"].max() + 1
]

labels_direccion = [
    "0 - 12 meses",
    "13 - 24 meses",
    "25 - 48 meses",
    "49 - 72 meses",
    "73 - 120 meses",
    "Más de 120 meses"
]

df["rango_antiguedad_direccion"] = pd.cut(
    df["antiguedad_direccion_meses"],
    bins=bins_direccion,
    labels=labels_direccion,
    include_lowest=True
)

conteo_direccion = (
    df["rango_antiguedad_direccion"]
    .value_counts()
    .sort_index()
)

# GRÁFICO 1
plt.figure(figsize=(11,5))

ax = sns.barplot(
    x=conteo_direccion.index,
    y=conteo_direccion.values
)

plt.title("Distribución de la antigüedad de la dirección")
plt.xlabel("Antigüedad de la dirección")
plt.ylabel("Cantidad de observaciones")

plt.xticks(rotation=20)

for i, valor in enumerate(conteo_direccion.values):
    ax.text(
        i,
        valor + 100,
        f"{valor:,}",
        ha="center"
    )

plt.tight_layout()
plt.show()


# GRÁFICO 2
plt.figure(figsize=(10,3))

sns.boxplot(
    data=df,
    x="antiguedad_direccion_meses"
)

plt.title("Detección de valores atípicos - Antigüedad de la dirección")
plt.xlabel("Antigüedad de la dirección (meses)")

plt.tight_layout()
plt.show()

**Visitas web últimos 90 días**

In [ ]:
# ============================================
# VISITAS WEB ÚLTIMOS 90 DÍAS
# ============================================

conteo_visitas = (
    df["visitas_web_ultimos_90_dias"]
    .value_counts()
    .sort_index()
)

# GRÁFICO 1
plt.figure(figsize=(10,5))

ax = sns.barplot(
    x=conteo_visitas.index,
    y=conteo_visitas.values
)

plt.title("Distribución de visitas web en los últimos 90 días")
plt.xlabel("Número de visitas")
plt.ylabel("Cantidad de observaciones")

plt.xticks(rotation=45)

for i, valor in enumerate(conteo_visitas.values):
    ax.text(
        i,
        valor + 30,
        f"{valor:,}",
        ha="center",
        fontsize=8
    )

plt.tight_layout()
plt.show()


# GRÁFICO 2
plt.figure(figsize=(10,3))

sns.boxplot(
    data=df,
    x="visitas_web_ultimos_90_dias"
)

plt.title("Detección de valores atípicos - Visitas web")
plt.xlabel("Visitas web en los últimos 90 días")

plt.tight_layout()
plt.show()

**Distancia a al sucursal**

In [ ]:
# ============================================
# DISTANCIA A LA SUCURSAL
# ============================================

grupos_distancia, limites_distancia = pd.cut(
    df["distancia_sucursal_km"],
    bins=5,
    retbins=True
)

etiquetas_distancia = []

for i in range(len(limites_distancia) - 1):
    etiquetas_distancia.append(
        f"{limites_distancia[i]:.1f} - {limites_distancia[i+1]:.1f} km"
    )

df["rango_distancia"] = pd.cut(
    df["distancia_sucursal_km"],
    bins=limites_distancia,
    labels=etiquetas_distancia,
    include_lowest=True
)

conteo_distancia = (
    df["rango_distancia"]
    .value_counts()
    .sort_index()
)

# GRÁFICO 1
plt.figure(figsize=(11,5))

ax = sns.barplot(
    x=conteo_distancia.index,
    y=conteo_distancia.values
)

plt.title("Distribución de la distancia a la sucursal")
plt.xlabel("Distancia a la sucursal")
plt.ylabel("Cantidad de observaciones")

plt.xticks(rotation=20)

for i, valor in enumerate(conteo_distancia.values):
    ax.text(
        i,
        valor + 100,
        f"{valor:,}",
        ha="center"
    )

plt.tight_layout()
plt.show()


# GRÁFICO 2
plt.figure(figsize=(10,3))

sns.boxplot(
    data=df,
    x="distancia_sucursal_km"
)

plt.title("Detección de valores atípicos - Distancia a la sucursal")
plt.xlabel("Distancia (km)")

plt.tight_layout()
plt.show()

**Dia preferido de pago**

In [ ]:
# ============================================
# DÍA PREFERIDO DE PAGO
# ============================================

conteo_pago = (
    df["dia_preferido_pago"]
    .value_counts()
    .sort_index()
)

# GRÁFICO 1
plt.figure(figsize=(12,5))

ax = sns.barplot(
    x=conteo_pago.index,
    y=conteo_pago.values
)

plt.title("Distribución del día preferido de pago")
plt.xlabel("Día del mes")
plt.ylabel("Cantidad de observaciones")

for i, valor in enumerate(conteo_pago.values):
    ax.text(
        i,
        valor + 20,
        f"{valor:,}",
        ha="center",
        fontsize=7
    )

plt.tight_layout()
plt.show()


# GRÁFICO 2
plt.figure(figsize=(10,3))

sns.boxplot(
    data=df,
    x="dia_preferido_pago"
)

plt.title("Distribución del día preferido de pago")
plt.xlabel("Día del mes")

plt.tight_layout()
plt.show()

**Días desde la última interacción**

In [ ]:
# ============================================
# DÍAS DESDE LA ÚLTIMA INTERACCIÓN
# ============================================

bins_interaccion = [0, 30, 60, 90, 180, 365]

labels_interaccion = [
    "0 - 30 días",
    "31 - 60 días",
    "61 - 90 días",
    "91 - 180 días",
    "181 - 365 días"
]

df["rango_ultima_interaccion"] = pd.cut(
    df["dias_ultima_interaccion"],
    bins=bins_interaccion,
    labels=labels_interaccion,
    include_lowest=True
)

conteo_interaccion = (
    df["rango_ultima_interaccion"]
    .value_counts()
    .sort_index()
)

# GRÁFICO 1
plt.figure(figsize=(10,5))

ax = sns.barplot(
    x=conteo_interaccion.index,
    y=conteo_interaccion.values
)

plt.title("Distribución de días desde la última interacción")
plt.xlabel("Días desde la última interacción")
plt.ylabel("Cantidad de observaciones")

for i, valor in enumerate(conteo_interaccion.values):
    ax.text(
        i,
        valor + 100,
        f"{valor:,}",
        ha="center"
    )

plt.tight_layout()
plt.show()


# GRÁFICO 2
plt.figure(figsize=(10,3))

sns.boxplot(
    data=df,
    x="dias_ultima_interaccion"
)

plt.title("Detección de valores atípicos - Última interacción")
plt.xlabel("Días desde la última interacción")

plt.tight_layout()
plt.show()

**Categoricas**

In [ ]:
variables_categoricas = [
    "ocupacion",
    "region",
    "canal_adquisicion",
    "banda_riesgo",
    "dispositivo_principal"
]

In [ ]:
def grafico_categoria(columna, titulo):

    conteo = df[columna].value_counts()

    plt.figure(figsize=(8,5))

    ax = sns.barplot(
        x=conteo.index,
        y=conteo.values
    )

    plt.title(titulo)
    plt.xlabel("")
    plt.ylabel("Cantidad de observaciones")

    plt.xticks(rotation=25)

    for i, valor in enumerate(conteo.values):
        ax.text(
            i,
            valor + 50,
            f"{valor:,}",
            ha="center"
        )

    plt.tight_layout()
    plt.show()

In [ ]:
grafico_categoria(
    "ocupacion",
    "Distribución según ocupación"
)

In [ ]:
grafico_categoria(
    "region",
    "Distribución según región"
)

In [ ]:
grafico_categoria(
    "canal_adquisicion",
    "Distribución según canal de adquisición"
)

In [ ]:
grafico_categoria(
    "banda_riesgo",
    "Distribución según banda de riesgo"
)

In [ ]:
grafico_categoria(
    "dispositivo_principal",
    "Distribución según dispositivo principal"
)

**Analisis de boleanos**

In [ ]:
variables_booleanas = [
    "tiene_tarjeta_credito",
    "activo_movil",
    "es_nuevo_cliente",
    "tiene_prestamo",
    "tiene_seguro"
]

In [ ]:
def grafico_booleana(columna, titulo):

    conteo = df[columna].value_counts().sort_index()

    plt.figure(figsize=(6,4))

    ax = sns.barplot(
        x=conteo.index.astype(str),
        y=conteo.values
    )

    plt.title(titulo)
    plt.xlabel("")
    plt.ylabel("Cantidad de observaciones")

    for i, valor in enumerate(conteo.values):
        ax.text(
            i,
            valor + 100,
            f"{valor:,}",
            ha="center"
        )

    plt.tight_layout()
    plt.show()

In [ ]:
grafico_booleana(
    "tiene_tarjeta_credito",
    "Distribución de clientes con tarjeta de crédito"
)

In [ ]:
grafico_booleana(
    "activo_movil",
    "Distribución según actividad móvil"
)

In [ ]:
grafico_booleana(
    "tiene_prestamo",
    "Distribución de clientes con préstamo"
)

In [ ]:
grafico_booleana(
    "tiene_seguro",
    "Distribución de clientes con seguro"
)

**ANALISIS BIVARIADO**

**Categoria vs objetivo**

In [ ]:
def conversion_categoria(columna, titulo):

    datos = (
        df.groupby(columna)["objetivo"]
          .mean()
          .mul(100)
          .sort_values(ascending=False)
          .reset_index()
    )

    plt.figure(figsize=(8,5))

    ax = sns.barplot(
        data=datos,
        x=columna,
        y="objetivo"
    )

    plt.title(titulo)
    plt.xlabel("")
    plt.ylabel("Tasa de conversión (%)")

    plt.xticks(rotation=25)

    # Porcentaje exacto
    for i, valor in enumerate(datos["objetivo"]):
        ax.text(
            i,
            valor + 0.2,
            f"{valor:.1f}%",
            ha="center"
        )

    plt.tight_layout()
    plt.show()

In [ ]:
conversion_categoria(
    "ocupacion",
    "Tasa de conversión según ocupación"
)

In [ ]:
conversion_categoria(
    "region",
    "Tasa de conversión según región"
)

In [ ]:
conversion_categoria(
    "canal_adquisicion",
    "Tasa de conversión según canal de adquisición"
)

In [ ]:
conversion_categoria(
    "banda_riesgo",
    "Tasa de conversión según banda de riesgo"
)

In [ ]:
conversion_categoria(
    "dispositivo_principal",
    "Tasa de conversión según dispositivo principal"
)

**Booleanas vs objetivo**

In [ ]:
def conversion_booleana(columna, titulo):

    datos = (
        df.groupby(columna)["objetivo"]
          .mean()
          .mul(100)
          .reset_index()
    )

    datos[columna] = datos[columna].map({
        False: "No",
        True: "Sí"
    })

    plt.figure(figsize=(6,5))

    ax = sns.barplot(
        data=datos,
        x=columna,
        y="objetivo"
    )

    plt.title(titulo)
    plt.xlabel("")
    plt.ylabel("Tasa de conversión (%)")

    for i, valor in enumerate(datos["objetivo"]):
        ax.text(
            i,
            valor + 0.2,
            f"{valor:.1f}%",
            ha="center"
        )

    plt.tight_layout()
    plt.show()

In [ ]:
conversion_booleana(
    "tiene_tarjeta_credito",
    "Conversión según tenencia de tarjeta de crédito"
)

In [ ]:
conversion_booleana(
    "activo_movil",
    "Conversión según actividad móvil"
)

In [ ]:
conversion_booleana(
    "es_nuevo_cliente",
    "Conversión según condición de nuevo cliente"
)

In [ ]:
conversion_booleana(
    "tiene_prestamo",
    "Conversión según tenencia de préstamo"
)

In [ ]:
conversion_booleana(
    "tiene_seguro",
    "Conversión según tenencia de seguro"
)

**NUMERICO VS OBJETIVO**

**Ingreso vs convencion**

In [ ]:
plt.figure(figsize=(8,5))

sns.boxplot(
    data=df,
    x="objetivo",
    y="ingresos"
)

plt.title("Ingresos según conversión")
plt.xlabel("Conversión")
plt.ylabel("Ingresos")

plt.xticks(
    [0,1],
    ["No convirtió", "Convirtió"]
)

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8,5))

sns.boxplot(
    data=df,
    x="objetivo",
    y="saldo_promedio"
)

plt.title("Saldo promedio según conversión")
plt.xlabel("Conversión")
plt.ylabel("Saldo promedio")

plt.xticks(
    [0,1],
    ["No convirtió", "Convirtió"]
)

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8,5))

sns.boxplot(
    data=df,
    x="objetivo",
    y="visitas_web_ultimos_90_dias"
)

plt.title("Visitas web según conversión")
plt.xlabel("Conversión")
plt.ylabel("Visitas web en los últimos 90 días")

plt.xticks(
    [0,1],
    ["No convirtió", "Convirtió"]
)

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8,5))

sns.boxplot(
    data=df,
    x="objetivo",
    y="dias_ultima_interaccion"
)

plt.title("Última interacción según conversión")
plt.xlabel("Conversión")
plt.ylabel("Días desde la última interacción")

plt.xticks(
    [0,1],
    ["No convirtió", "Convirtió"]
)

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8,5))

sns.boxplot(
    data=df,
    x="objetivo",
    y="dias_ultima_transaccion"
)

plt.title("Última transacción según conversión")
plt.xlabel("Conversión")
plt.ylabel("Días desde la última transacción")

plt.xticks(
    [0,1],
    ["No convirtió", "Convirtió"]
)

plt.tight_layout()
plt.show()

In [ ]:
# Calcular tasa de conversión por mes
conversion_mes = (
    df.groupby("mes")["objetivo"]
      .mean()
      .mul(100)
      .reset_index()
)

# Crear etiquetas de mes entendibles
nombres_meses = {
    202601: "Ene",
    202602: "Feb",
    202603: "Mar",
    202604: "Abr",
    202605: "May",
    202606: "Jun",
    202607: "Jul",
    202608: "Ago",
    202609: "Sep",
    202610: "Oct",
    202611: "Nov"
}

conversion_mes["mes_nombre"] = conversion_mes["mes"].map(nombres_meses)

# Gráfico
plt.figure(figsize=(11,5))

plt.plot(
    conversion_mes["mes_nombre"],
    conversion_mes["objetivo"],
    marker="o"
)

plt.title("Tasa de conversión por mes", pad=20)
plt.xlabel("Mes")
plt.ylabel("Tasa de conversión (%)")

# Mostrar porcentaje exacto
for x, y in zip(
    conversion_mes["mes_nombre"],
    conversion_mes["objetivo"]
):
    plt.text(
        x,
        y + 0.08,
        f"{y:.1f}%",
        ha="center",
        fontsize=9
    )

plt.ylim(
    conversion_mes["objetivo"].min() - 0.3,
    conversion_mes["objetivo"].max() + 0.5
)

plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()